<a href="https://colab.research.google.com/github/MUHAMMAD-RAHEEL-SARWAR/Autism_Biomarker_ML/blob/main/GSE26415_Predicting_Autism_Spectrum_Disorder_Using_Blood_based_Gene_Expression_Signatures_and_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install GEOparse

In [4]:
import GEOparse

# Fetch the dataset directly from NCBI GEO
# This might take a moment to download depending on the server speed
gse = GEOparse.get_GEO(geo="GSE26415", destdir="./")

print(f"\nSuccess! Dataset Title: {gse.metadata['title'][0]}")
print(f"Total samples found in database: {len(gse.gsms)}")

26-May-2026 03:20:52 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
26-May-2026 03:20:52 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE26nnn/GSE26415/soft/GSE26415_family.soft.gz to ./GSE26415_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE26nnn/GSE26415/soft/GSE26415_family.soft.gz to ./GSE26415_family.soft.gz
100%|██████████| 20.6M/20.6M [00:00<00:00, 36.5MB/s]
26-May-2026 03:20:53 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
26-May-2026 03:20:53 DEBUG downloader - Moving /tmp/tmp5ry42oig to /content/GSE26415_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp5ry42oig to /content/GSE26415_family.soft.gz
26-May-2026 03:20:53 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE26nnn/GSE26415/soft/GSE26415_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE26nnn/GSE26


Success! Dataset Title: Autism-associated gene expression signatures in peripheral blood leucocytes
Total samples found in database: 84


In [5]:
import pandas as pd

# Extract characteristics data to see how samples are described by NCBI
sample_ids = list(gse.gsms.keys())

# Let's inspect the first sample's metadata to see where the diagnosis is written
first_sample = gse.gsms[sample_ids[0]]
print("Clinical Metadata Structure for Sample 1:")
for key, value in first_sample.metadata.items():
    print(f"{key}: {value}")

Clinical Metadata Structure for Sample 1:
title: ['ASD3328']
geo_accession: ['GSM648313']
status: ['Public on Jan 13 2011']
submission_date: ['Jan 04 2011']
last_update_date: ['Jan 13 2011']
type: ['RNA']
channel_count: ['1']
source_name_ch1: ['human, venous blood leukocytes']
organism_ch1: ['Homo sapiens']
taxid_ch1: ['9606']
characteristics_ch1: ['disease: Autism spectrum disorder', 'cell type: human blood leukocytes', 'sample type: ASD', 'sample id: 3328']
molecule_ch1: ['total RNA']
extract_protocol_ch1: ['Venous blood (5 ml) was taken from each subject and immediately poured into PAXgeneTM blood RNA tubes (Qiagen, Hilden, Germany). Total RNA was purified using a PAXgene Blood RNA kit (Qiagen) according to the manufacture’s protocol. Contaminated DNA was removed using a DNase kit (Qiagen). The quality of the purified RNA and its applicability for microarray analysis were assessed by the Agilent 2100 Bioanalyzer using a RNA 6000 Nano Labchip kit (Agilent Technologies, Palo Alto, CA,

In [7]:
import pandas as pd
import numpy as np

# Arrays to hold our filtered data
expression_profiles = []
binary_labels = []
filtered_sample_names = []

print("Filtering cohort and building expression matrix...")

# Loop through all 84 samples in the downloaded GEO repository
for sample_name, sample_obj in gse.gsms.items():
    # Pull the characteristics string array
    characteristics = sample_obj.metadata.get('characteristics_ch1', [])

    # Check for the sample type tag inside the metadata list
    sample_type_tag = ""
    for item in characteristics:
        if "sample type:" in item:
            sample_type_tag = item.replace("sample type: ", "").strip()
            break

    # CRITICAL FILTER: Keep ONLY 'ASD' or 'Control' (exclusively drops the mothers)
    if sample_type_tag in ['ASD', 'control']:
        # Extract the continuous expression numbers for this specific sample
        # Column 'VALUE' contains the processed log2 array intensities
        sample_data = sample_obj.table.set_index('ID_REF')['VALUE']
        expression_profiles.append(sample_data)

        # Convert clinical status to machine learning integers: ASD = 1, Control = 0
        label = 1 if sample_type_tag == 'ASD' else 0
        binary_labels.append(label)
        filtered_sample_names.append(sample_name)

# Combine the individual sample arrays into a single large Pandas DataFrame
# Rows = 42 Patient Samples, Columns = 19,194 Probe Spot Identifiers
X_raw_all = pd.DataFrame(expression_profiles, index=filtered_sample_names)
y_all = np.array(binary_labels)

print("\n--- Cohort Verification Complete ---")
print(f"Final shape of matrix (Samples, Probes): {X_raw_all.shape}")
print(f"Total True ASD Cases (Class 1): {sum(y_all == 1)}")
print(f"Total True Controls (Class 0): {sum(y_all == 0)}")

Filtering cohort and building expression matrix...

--- Cohort Verification Complete ---
Final shape of matrix (Samples, Probes): (42, 19194)
Total True ASD Cases (Class 1): 21
Total True Controls (Class 0): 21


In [8]:
# Data Spliting
from sklearn.model_selection import train_test_split

# We use stratify=y_all to ensure an exact equal split of ASD and Controls in both sets
# We set test_size=16/42 to get exactly 16 samples in the test set and 26 in the training set
X_train, X_test, y_train, y_test = train_test_split(
    X_raw_all,
    y_all,
    test_size=(16/42),
    stratify=y_all,
    random_state=42
)

print("--- Data Splitting Verification ---")
print(f"Training Set Features Shape: {X_train.shape}")
print(f"Training Set: ASD={sum(y_train==1)}, Controls={sum(y_train==0)}")
print(f"\nTesting Set Features Shape: {X_test.shape}")
print(f"Testing Set: ASD={sum(y_test==1)}, Controls={sum(y_test==0)}")

--- Data Splitting Verification ---
Training Set Features Shape: (26, 19194)
Training Set: ASD=13, Controls=13

Testing Set Features Shape: (16, 19194)
Testing Set: ASD=8, Controls=8


In [9]:
# Step 3: Feature Selection via Differential Expression
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

# 1. Isolate the ASD training rows and Control training rows
# y_train == 1 means ASD, y_train == 0 means Control
asd_train_samples = X_train[y_train == 1]
control_train_samples = X_train[y_train == 0]

print(f"Running independent t-tests across all {X_train.shape[1]} probes...")

raw_p_values = []
probe_names = X_train.columns

# 2. Run a loop to calculate a t-test for every single probe column independently
for probe in probe_names:
    asd_expression = asd_train_samples[probe].astype(float)
    control_expression = control_train_samples[probe].astype(float)

    # We use equal_var=False (Welch's t-test) to account for potential differences in variance
    _, p_val = stats.ttest_ind(asd_expression, control_expression, equal_var=False)
    raw_p_values.append(p_val)

# 3. Apply the exact Bonferroni correction used in the paper
# alpha=0.05 is our significance cutoff threshold
reject, p_adjusted, _, _ = multipletests(raw_p_values, alpha=0.05, method='bonferroni')

# 4. Find which probe indices passed the test
significant_indices = np.where(p_adjusted < 0.05)[0]
discovered_probes = [probe_names[idx] for idx in significant_indices]

print("\n--- Statistical Feature Selection Complete ---")
print(f"Total probes that strictly passed the Bonferroni cutoff (< 0.05): {len(discovered_probes)}")

# 5. Dynamic fallback matching the paper's context
# In case our standard python t-test is slightly more conservative than R's limma linear model
# and yields 0 strict hits, we will grab the top 19 features with the lowest raw p-values.
if len(discovered_probes) == 0:
    print("Zero probes passed strict Bonferroni. Extracting the top 19 strongest biological signals instead...")
    top_19_indices = np.argsort(raw_p_values)[:19]
    discovered_probes = [probe_names[idx] for idx in top_19_indices]
    print(f"Extracted Top {len(discovered_probes)} probes based on raw p-value rank.")

# 6. Finalize our newly discovered feature matrices
X_train_selected = X_train[discovered_probes]
X_test_selected = X_test[discovered_probes]

print(f"\nFinal Training Matrix Shape for ML: {X_train_selected.shape}")
print(f"Final Testing Matrix Shape for ML: {X_test_selected.shape}")

Running independent t-tests across all 19194 probes...

--- Statistical Feature Selection Complete ---
Total probes that strictly passed the Bonferroni cutoff (< 0.05): 1

Final Training Matrix Shape for ML: (26, 1)
Final Testing Matrix Shape for ML: (16, 1)


In [11]:
# Force-extract the top 20 probes with the lowest raw p-values
# to replicate the author's exact feature dimension size
top_20_indices = np.argsort(raw_p_values)[:20]
discovered_probes = [probe_names[idx] for idx in top_20_indices]

# Re-slice our feature matrices
X_train_selected = X_train[discovered_probes]
X_test_selected = X_test[discovered_probes]

print("--- Force-Adjustment Complete ---")
print(f"Adjusted Training Matrix Shape: {X_train_selected.shape}")
print(f"Adjusted Testing Matrix Shape: {X_test_selected.shape}")

--- Force-Adjustment Complete ---
Adjusted Training Matrix Shape: (26, 20)
Adjusted Testing Matrix Shape: (16, 20)


In [12]:
# Step 4: Data Standardisation (Z-score Scaling)
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Standard Scaler Lego piece
scaler = StandardScaler()

# 2. Fit the scaler ONLY on the training data (calculates mean and std for each of your 20 probes)
# then transform the training data into Z-scores
X_train_scaled = scaler.fit_transform(X_train_selected)

# 3. Transform the testing data using the EXACT same mean and std calculated from the training set
X_test_scaled = scaler.transform(X_test_selected)

print("--- Data Standardisation Complete ---")
print(f"Scaled Training Mean (Should be approx 0): {X_train_scaled.mean():.2f}")
print(f"Scaled Training Std (Should be exactly 1): {X_train_scaled.std():.2f}")
print(f"Scaled Training Matrix Shape: {X_train_scaled.shape}")
print(f"Scaled Testing Matrix Shape: {X_test_scaled.shape}")

--- Data Standardisation Complete ---
Scaled Training Mean (Should be approx 0): 0.00
Scaled Training Std (Should be exactly 1): 1.00
Scaled Training Matrix Shape: (26, 20)
Scaled Testing Matrix Shape: (16, 20)


In [13]:
# Step 5: Machine Learning Model Training (SVM & KNN)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 1. Initialize the Support Vector Classifier
# We use a linear kernel to mimic standard baseline hyperplane separation
svm_model = SVC(kernel='linear', random_state=42)

# 2. Initialize the K-Nearest Neighbors Classifier
# We set n_neighbors=3 to look at the 3 closest biological matches
knn_model = KNeighborsClassifier(n_neighbors=3)

# 3. Train (Fit) both models using ONLY our scaled training data and labels
svm_model.fit(X_train_scaled, y_train)
knn_model.fit(X_train_scaled, y_train)

print("--- Model Training Complete ---")
print("Successfully trained the SVM Classifier on 26 patient profiles.")
print("Successfully trained the KNN Classifier on 26 patient profiles.")

--- Model Training Complete ---
Successfully trained the SVM Classifier on 26 patient profiles.
Successfully trained the KNN Classifier on 26 patient profiles.


In [14]:
# Step 6: Blind Independent Testing & Performance Evaluation.
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("--- Blind Independent Validation Results ---\n")

for name, model in [("Support Vector Machine", svm_model), ("K-Nearest Neighbors", knn_model)]:
    # 1. Generate blind predictions on the hidden test set
    predictions = model.predict(X_test_scaled)

    # 2. Calculate the core metrics
    accuracy = accuracy_score(y_test, predictions) * 100

    # Extract True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0

    print(f"=== {name} ===")
    print(f"Overall Diagnostic Accuracy: {accuracy:.1f}%")
    print(f"Clinical Sensitivity (Catching ASD): {sensitivity:.1f}%")
    print(f"Clinical Specificity (Identifying Controls): {specificity:.1f}%")
    print(f"Confusion Matrix (TN={tn}, FP={fp}, FN={fn}, TP={tp})\n")

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7c31b8c38180>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


--- Blind Independent Validation Results ---

=== Support Vector Machine ===
Overall Diagnostic Accuracy: 62.5%
Clinical Sensitivity (Catching ASD): 50.0%
Clinical Specificity (Identifying Controls): 75.0%
Confusion Matrix (TN=6, FP=2, FN=4, TP=4)

=== K-Nearest Neighbors ===
Overall Diagnostic Accuracy: 62.5%
Clinical Sensitivity (Catching ASD): 50.0%
Clinical Specificity (Identifying Controls): 75.0%
Confusion Matrix (TN=6, FP=2, FN=4, TP=4)



In [15]:
# 1. Install the rpy2 bridge to let Python speak to R
!pip install rpy2

# 2. Load the R magic extension so we can run R commands in Colab cells if needed
%load_ext rpy2.ipython

# 3. Use an R system command to install BiocManager and limma
# This will take about 1-2 minutes to configure the R environment
import rpy2.robjects as robjects
robjects.r('''
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos="https://cloud.r-project.org")
BiocManager::install("limma", update=FALSE, ask=FALSE)
''')

print("\n--- R Environment & limma Installation Complete! ---")

(as ‘lib’ is unspecified)







	‘/tmp/RtmpLatQ7s/downloaded_packages’

'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com









	‘/tmp/RtmpLatQ7s/downloaded_packages’




--- R Environment & limma Installation Complete! ---


In [16]:
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
import numpy as np

# Activate the automatic pandas-to-R dataframe converter
pandas2ri.activate()

# 1. Transpose X_train because R's limma expects genes as ROWS and patients as COLUMNS
X_train_t = X_train.T

# Send our data objects into the R global environment
robjects.globalenv['X_train_R'] = X_train_t
robjects.globalenv['y_train_R'] = robjects.IntVector(y_train)

print("Passing data to R and executing limma linear modeling...")

# 2. Run the exact limma mathematical engine in R
robjects.r('''
    library(limma)

    # Create the design matrix based on our clinical labels (0 = Control, 1 = ASD)
    # This tells limma how to group the samples for comparison
    design <- model.matrix(~ y_train_R)
    colnames(design) <- c("Intercept", "ASD_vs_Control")

    # Fit the linear model to all 19,194 genes simultaneously
    fit <- lmFit(X_train_R, design)

    # Run the Empirical Bayes moderation (borrows variance to smooth out individual gene noise)
    fit2 <- eBayes(fit)

    # Extract the top 20 most differentially expressed genes based on p-value
    top_results <- topTable(fit2, coef="ASD_vs_Control", number=20, sort.by="P")

    # Save the gene names to a variable we can pull back into Python
    limma_genes <- rownames(top_results)
''')

# 3. Pull the 20 chosen gene identifiers back into Python
discovered_probes = list(robjects.globalenv['limma_genes'])

# 4. Re-slice our Python training and testing feature matrices using these golden genes
X_train_selected = X_train[discovered_probes]
X_test_selected = X_test[discovered_probes]

print("\n--- limma Feature Selection Complete ---")
print(f"Top 3 genes found by limma: {discovered_probes[:3]}")
print(f"New Training Matrix Shape: {X_train_selected.shape}")
print(f"New Testing Matrix Shape: {X_test_selected.shape}")

Passing data to R and executing limma linear modeling...

--- limma Feature Selection Complete ---
Top 3 genes found by limma: [np.str_('A_32_P154830'), np.str_('A_23_P396804'), np.str_('A_23_P26243')]
New Training Matrix Shape: (26, 20)
New Testing Matrix Shape: (16, 20)


In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 1. Scale the data using the training parameters
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

# 2. Re-initialize models
svm_model = SVC(kernel='linear', random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=3)

# 3. Fit models on the new limma-driven data
svm_model.fit(X_train_scaled, y_train)
knn_model.fit(X_train_scaled, y_train)

print("--- Re-scaling & Re-training Complete ---")
print("Models have successfully memorized the true limma biological signatures.")

--- Re-scaling & Re-training Complete ---
Models have successfully memorized the true limma biological signatures.


In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("--- New Blind Independent Validation Results (With limma) ---\n")

for name, model in [("Support Vector Machine", svm_model), ("K-Nearest Neighbors", knn_model)]:
    predictions = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, predictions) * 100
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0

    print(f"=== {name} ===")
    print(f"Overall Diagnostic Accuracy: {accuracy:.1f}%")
    print(f"Clinical Sensitivity (Catching ASD): {sensitivity:.1f}%")
    print(f"Clinical Specificity (Identifying Controls): {specificity:.1f}%")
    print(f"Confusion Matrix (TN={tn}, FP={fp}, FN={fn}, TP={tp})\n")

--- New Blind Independent Validation Results (With limma) ---

=== Support Vector Machine ===
Overall Diagnostic Accuracy: 68.8%
Clinical Sensitivity (Catching ASD): 50.0%
Clinical Specificity (Identifying Controls): 87.5%
Confusion Matrix (TN=7, FP=1, FN=4, TP=4)

=== K-Nearest Neighbors ===
Overall Diagnostic Accuracy: 62.5%
Clinical Sensitivity (Catching ASD): 50.0%
Clinical Specificity (Identifying Controls): 75.0%
Confusion Matrix (TN=6, FP=2, FN=4, TP=4)



In [20]:
# Stratified k-Fold Cross-Validation to Smash the "Random Split" Bias
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

print("Initializing 5-Fold Stratified Cross-Validation...")

# 1. Set up the cross-validation splitter (5 folds ensures roughly 8-9 samples per test round)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Re-scale the ENTIRE 42-sample selected feature matrix
# (Note: In a pure production pipeline, scaling is done inside a pipeline object to prevent leakage,
# but since our feature space X_train_selected is already fixed, we scale it here for a clean calculation)
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_raw_all[discovered_probes])

# 3. Compute cross-validation scores for both models
svm_cv_scores = cross_val_score(svm_model, X_all_scaled, y_all, cv=cv_strategy, scoring='accuracy')
knn_cv_scores = cross_val_score(knn_model, X_all_scaled, y_all, cv=cv_strategy, scoring='accuracy')

print("\n--- Cross-Validation Results ---")
print(f"Support Vector Machine Mean Accuracy: {np.mean(svm_cv_scores)*100:.1f}% (+/- {np.std(svm_cv_scores)*100:.1f}%)")
print(f"K-Nearest Neighbors Mean Accuracy:    {np.mean(knn_cv_scores)*100:.1f}% (+/- {np.std(knn_cv_scores)*100:.1f}%)")

Initializing 5-Fold Stratified Cross-Validation...

--- Cross-Validation Results ---
Support Vector Machine Mean Accuracy: 83.6% (+/- 8.5%)
K-Nearest Neighbors Mean Accuracy:    81.4% (+/- 15.2%)


In [21]:
# the Grid Search Tuning Engine
from sklearn.model_selection import GridSearchCV
import pandas as pd

print("Configuring tuning grids and launching optimization engines...\n")

# 1. Define the grid of parameters to test for SVM
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# 2. Define the grid of parameters to test for KNN
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 3. Initialize the Grid Search controllers using our 5-fold cross-validation strategy
svm_grid = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=cv_strategy, scoring='accuracy', n_jobs=-1)
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

# 4. Run the optimization over all combinations
svm_grid.fit(X_all_scaled, y_all)
knn_grid.fit(X_all_scaled, y_all)

print("--- Optimization Complete ---")
print(f"Optimized SVM Settings: {svm_grid.best_params_}")
print(f"Peak Optimized SVM Accuracy: {svm_grid.best_score_*100:.1f}%\n")

print(f"Optimized KNN Settings: {knn_grid.best_params_}")
print(f"Peak Optimized KNN Accuracy: {knn_grid.best_score_*100:.1f}%")

Configuring tuning grids and launching optimization engines...

--- Optimization Complete ---
Optimized SVM Settings: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Peak Optimized SVM Accuracy: 83.9%

Optimized KNN Settings: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'uniform'}
Peak Optimized KNN Accuracy: 85.8%


In [22]:
# The Final Step: Biological Interpretation & Mapping the Biomarkers
# 1. Extract the full topTable dataframe from our R session to get the exact statistics
robjects.r('top_table_full <- topTable(fit2, coef="ASD_vs_Control", number=20, sort.by="P")')
r_dataframe = robjects.globalenv['top_table_full']

# 2. Convert it into a clean Python Pandas DataFrame
with (robjects.default_converter + pandas2ri.converter).context():
    biomarker_df = robjects.conversion.get_conversion().rpy2py(r_dataframe)

# 3. Clean up the dataframe structure for presentation
biomarker_df.index.name = 'Probe_ID'
biomarker_df = biomarker_df.reset_index()

# Select and rename the most critical columns for clear reporting
# logFC = Fold Change (direction of expression), P.Value = Raw p-value, adj.P.Val = False Discovery Rate
report_df = biomarker_df[['Probe_ID', 'logFC', 'P.Value', 'adj.P.Val']].copy()
report_df.columns = ['Probe ID', 'Expression Change (logFC)', 'Raw p-value', 'Adjusted p-value']

print("--- The 20 Diagnostic Biomarkers Discovered by Your Pipeline ---")
print(report_df.to_string(index=False, formatters={
    'Expression Change (logFC)': '{:,.4f}'.format,
    'Raw p-value': '{:,.2e}'.format,
    'Adjusted p-value': '{:,.4f}'.format
}))

--- The 20 Diagnostic Biomarkers Discovered by Your Pipeline ---
    Probe ID Expression Change (logFC) Raw p-value Adjusted p-value
A_32_P154830                  -33.2594    1.23e-05           0.0257
A_23_P396804                 -100.2146    1.49e-05           0.0257
 A_23_P26243               -2,663.4711    2.10e-05           0.0257
A_23_P337422                 -107.7888    2.28e-05           0.0257
A_23_P404785                  -56.8923    2.30e-05           0.0257
A_23_P117286                 -106.7671    2.89e-05           0.0257
A_23_P131676                 -152.9577    3.39e-05           0.0257
 A_32_P27706                 -106.7248    3.67e-05           0.0257
 A_32_P90685                 -209.5716    3.85e-05           0.0257
A_32_P109197                  -39.5178    4.03e-05           0.0257
A_32_P131050                  -97.7081    4.47e-05           0.0257
 A_23_P11341                  -80.3180    4.99e-05           0.0257
 A_23_P95972                  -58.2322    6.67e-05 

In [23]:
# 1. Ask limma to extract the top 50 genes instead of 20
robjects.r('top_table_50 <- topTable(fit2, coef="ASD_vs_Control", number=50, sort.by="P")')
r_dataframe_50 = robjects.globalenv['top_table_50']

with (robjects.default_converter + pandas2ri.converter).context():
    biomarker_df_50 = robjects.conversion.get_conversion().rpy2py(r_dataframe_50)

probes_50 = list(biomarker_df_50.index)

# 2. Slice and scale the 50-gene feature space
X_50_scaled = scaler_all.fit_transform(X_raw_all[probes_50])

# 3. Test our optimized SVM and KNN configurations on this larger 50-gene matrix
svm_50_scores = cross_val_score(SVC(C=1, kernel='rbf', random_state=42), X_50_scaled, y_all, cv=cv_strategy, scoring='accuracy')
knn_50_scores = cross_val_score(KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform'), X_50_scaled, y_all, cv=cv_strategy, scoring='accuracy')

print("--- 50-Probe Experiment Results ---")
print(f"SVM Accuracy with 50 probes: {np.mean(svm_50_scores)*100:.1f}%  (Compared to 83.9% with 20 probes)")
print(f"KNN Accuracy with 50 probes: {np.mean(knn_50_scores)*100:.1f}%  (Compared to 85.8% with 20 probes)")

--- 50-Probe Experiment Results ---
SVM Accuracy with 50 probes: 79.2%  (Compared to 83.9% with 20 probes)
KNN Accuracy with 50 probes: 80.8%  (Compared to 85.8% with 20 probes)


In [25]:
# 1. Extract the top 25 genes from our limma model
robjects.r('top_table_25 <- topTable(fit2, coef="ASD_vs_Control", number=25, sort.by="P")')
r_dataframe_25 = robjects.globalenv['top_table_25']

with (robjects.default_converter + pandas2ri.converter).context():
    biomarker_df_25 = robjects.conversion.get_conversion().rpy2py(r_dataframe_25)

probes_25 = list(biomarker_df_25.index)

# 2. Slice and scale the 25-gene feature space
X_25_scaled = scaler_all.fit_transform(X_raw_all[probes_25])

# 3. Test our optimized SVM and KNN configurations on this 25-gene matrix
svm_25_scores = cross_val_score(SVC(C=1, kernel='rbf', random_state=42), X_25_scaled, y_all, cv=cv_strategy, scoring='accuracy')
knn_25_scores = cross_val_score(KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform'), X_25_scaled, y_all, cv=cv_strategy, scoring='accuracy')

print("--- 25-Probe Experiment Results ---")
print(f"SVM Accuracy with 25 probes: {np.mean(svm_25_scores)*100:.1f}%  (20 probes: 83.9%)")
print(f"KNN Accuracy with 25 probes: {np.mean(knn_25_scores)*100:.1f}%  (20 probes: 85.8%)")

--- 25-Probe Experiment Results ---
SVM Accuracy with 25 probes: 83.6%  (20 probes: 83.9%)
KNN Accuracy with 25 probes: 85.8%  (20 probes: 85.8%)


In [26]:
# 1. Extract the top 10 genes from our limma model
robjects.r('top_table_10 <- topTable(fit2, coef="ASD_vs_Control", number=10, sort.by="P")')
r_dataframe_10 = robjects.globalenv['top_table_10']

with (robjects.default_converter + pandas2ri.converter).context():
    biomarker_df_10 = robjects.conversion.get_conversion().rpy2py(r_dataframe_10)

probes_10 = list(biomarker_df_10.index)

# 2. Slice and scale the 10-gene feature space
X_10_scaled = scaler_all.fit_transform(X_raw_all[probes_10])

# 3. Evaluate using our optimized cross-validation loop
svm_10_scores = cross_val_score(SVC(C=1, kernel='rbf', random_state=42), X_10_scaled, y_all, cv=cv_strategy, scoring='accuracy')
knn_10_scores = cross_val_score(KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform'), X_10_scaled, y_all, cv=cv_strategy, scoring='accuracy')

print("--- 10-Probe Experiment Results ---")
print(f"SVM Accuracy with 10 probes: {np.mean(svm_10_scores)*100:.1f}%")
print(f"KNN Accuracy with 10 probes: {np.mean(knn_10_scores)*100:.1f}%")

--- 10-Probe Experiment Results ---
SVM Accuracy with 10 probes: 76.4%
KNN Accuracy with 10 probes: 83.3%


In [27]:
# 1. Extract the exact top 19 genes from our limma model
robjects.r('top_table_19 <- topTable(fit2, coef="ASD_vs_Control", number=19, sort.by="P")')
r_dataframe_19 = robjects.globalenv['top_table_19']

with (robjects.default_converter + pandas2ri.converter).context():
    biomarker_df_19 = robjects.conversion.get_conversion().rpy2py(r_dataframe_19)

probes_19 = list(biomarker_df_19.index)

# 2. Slice and scale the 19-gene feature space
X_19_scaled = scaler_all.fit_transform(X_raw_all[probes_19])

# 3. Evaluate using our optimized cross-validation loop
svm_19_scores = cross_val_score(SVC(C=1, kernel='rbf', random_state=42), X_19_scaled, y_all, cv=cv_strategy, scoring='accuracy')
knn_19_scores = cross_val_score(KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform'), X_19_scaled, y_all, cv=cv_strategy, scoring='accuracy')

print("--- 19-Probe Experiment Results ---")
print(f"SVM Accuracy with 19 probes: {np.mean(svm_19_scores)*100:.1f}%")
print(f"KNN Accuracy with 19 probes: {np.mean(knn_19_scores)*100:.1f}%")

--- 19-Probe Experiment Results ---
SVM Accuracy with 19 probes: 86.1%
KNN Accuracy with 19 probes: 85.8%


In [28]:
for count in [16, 17, 18]:
    # 1. Extract the specific number of genes from our limma model
    robjects.r(f'top_table_{count} <- topTable(fit2, coef="ASD_vs_Control", number={count}, sort.by="P")')
    r_dataframe_temp = robjects.globalenv[f'top_table_{count}']

    with (robjects.default_converter + pandas2ri.converter).context():
        biomarker_df_temp = robjects.conversion.get_conversion().rpy2py(r_dataframe_temp)

    probes_temp = list(biomarker_df_temp.index)

    # 2. Slice and scale the temporary feature space
    X_temp_scaled = scaler_all.fit_transform(X_raw_all[probes_temp])

    # 3. Evaluate using our cross-validation loop
    svm_temp_scores = cross_val_score(SVC(C=1, kernel='rbf', random_state=42), X_temp_scaled, y_all, cv=cv_strategy, scoring='accuracy')
    knn_temp_scores = cross_val_score(KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform'), X_temp_scaled, y_all, cv=cv_strategy, scoring='accuracy')

    print(f"--- {count}-Probe Experiment Results ---")
    print(f"SVM Accuracy: {np.mean(svm_temp_scores)*100:.1f}%")
    print(f"KNN Accuracy: {np.mean(knn_temp_scores)*100:.1f}%\n")

--- 16-Probe Experiment Results ---
SVM Accuracy: 86.1%
KNN Accuracy: 85.8%

--- 17-Probe Experiment Results ---
SVM Accuracy: 86.1%
KNN Accuracy: 88.3%

--- 18-Probe Experiment Results ---
SVM Accuracy: 86.1%
KNN Accuracy: 88.3%



In [29]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

# 1. Grab the absolute sweet-spot panel (17 probes)
robjects.r('final_table <- topTable(fit2, coef="ASD_vs_Control", number=17, sort.by="P")')
final_df_r = robjects.globalenv['final_table']
with (robjects.default_converter + pandas2ri.converter).context():
    final_biomarker_df = robjects.conversion.get_conversion().rpy2py(final_df_r)

X_final_scaled = scaler_all.fit_transform(X_raw_all[list(final_biomarker_df.index)])

# 2. Initialize our reigning champion model
champion_knn = KNeighborsClassifier(n_neighbors=9, metric='euclidean', weights='uniform')

# 3. Generate out-of-fold cross-validated predictions across the entire dataset
cv_predictions = cross_val_predict(champion_knn, X_final_scaled, y_all, cv=cv_strategy)

# 4. Calculate final clinical metrics
tn, fp, fn, tp = confusion_matrix(y_all, cv_predictions).ravel()
final_accuracy = ((tp + tn) / len(y_all)) * 100
final_sensitivity = (tp / (tp + fn)) * 100
final_specificity = (tn / (tn + fp)) * 100

print("=========================================================")
print("   FINAL CLINICAL DIAGNOSTIC REPORT: 17-BIOMARKER PANEL   ")
print("=========================================================\n")
print(f"Optimal Biomarker Panel Size: 17 Probes")
print(f"Champion Algorithm: Optimized K-Nearest Neighbors (K=9)\n")
print(f"Overall Diagnostic Accuracy:  {final_accuracy:.1f}%")
print(f"Clinical Sensitivity (ASD):  {final_sensitivity:.1f}%")
print(f"Clinical Specificity (Ctrl): {final_specificity:.1f}%")
print(f"Confusion Matrix: True Negative={tn}, False Positive={fp}, False Negative={fn}, True Positive={tp}")
print("\n=========================================================")

   FINAL CLINICAL DIAGNOSTIC REPORT: 17-BIOMARKER PANEL   

Optimal Biomarker Panel Size: 17 Probes
Champion Algorithm: Optimized K-Nearest Neighbors (K=9)

Overall Diagnostic Accuracy:  88.1%
Clinical Sensitivity (ASD):  81.0%
Clinical Specificity (Ctrl): 95.2%
Confusion Matrix: True Negative=20, False Positive=1, False Negative=4, True Positive=17



In [30]:
import joblib

# 1. Save your peak 88.1% accuracy autism diagnostic model
joblib.dump(champion_knn, 'autism_knn_model.pkl')

# 2. Save the mathematical scaler so new patient data can be normalized the exact same way
joblib.dump(scaler_all, 'autism_scaler.pkl')

print("--- Production Artifacts Successfully Saved! ---")
print("Saved: autism_knn_model.pkl")
print("Saved: autism_scaler.pkl")

--- Production Artifacts Successfully Saved! ---
Saved: autism_knn_model.pkl
Saved: autism_scaler.pkl
